# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signals checked first, each with a bucket table and n (see code below):**

1. **Staleness** (`days_since_last_update`) — the signal behind FlyRank's real
   `stale_visible_page` refresh flag. **Verdict: MIXED.** Decline rate rises across the first
   three buckets (0-30d: 51.1%, n=20,480 → 31-90d: 58.9%, n=175 → 91-180d: 61.1%, n=9,171), which
   supports the idea — but the stalest bucket reverses (180d+: 47.1%, n=174). That reversal is a
   small, thin bucket, so I'm not throwing staleness away, but I'm not building the rule on it
   alone either — a clean confirmation this is not.
2. **CTR-vs-position** (among visible pages ranked 1-20) — the signal behind FlyRank's real
   `needs_ctr_fix` flag. **Verdict: CONFIRMED.** Decline rate falls cleanly as CTR improves
   (<0.5%: 64.7%, n=12,203 → 0.5-1.5%: 50.7%, n=2,604 → 1.5-3%: 44.7%, n=244), across a
   well-populated majority of the eligible cohort. A small 3%+ bucket bounces back up, but at
   n=40 that's noise, not a contradiction.

**The rule, in plain words:** *A well-positioned, genuinely visible page whose click-through
rate is far below what its position should earn is worth a CTR review.* I'm anchoring on the
CONFIRMED signal (CTR-vs-position), not the MIXED one (staleness) — that MIXED verdict just
saved this rule from leaning on a signal that doesn't hold up cleanly.

**Score (no fitted weights, every term readable):**
`eligible = (1 <= avg_position <= 20) AND (impressions_90d >= 100)`
`score = eligible * (0.4 * visibility_score + 0.3 * position_quality_score + 0.3 * ctr_gap_score)`

**Reason code (one, applied to every flagged row):** `low_ctr_well_positioned`

**Action label:** `review_ctr`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# --- Signal 1: staleness (behind the real stale_visible_page refresh flag) ---
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=[-1, 30, 90, 180, 100000],
                                 labels=['0-30d', '31-90d', '91-180d', '180d+'])
signal1 = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'size'), decline_rate=('is_declining_label', 'mean')
)
print("SIGNAL 1 -- staleness vs decline rate  ->  verdict: MIXED")
print(signal1.round(3))
print()

# --- Signal 2: CTR-vs-position (behind the real needs_ctr_fix flag) ---
visible_ranked = df[(df['avg_position'] > 0) & (df['avg_position'] <= 20)
                     & (df['impressions_90d'] >= 100)].copy()
visible_ranked['ctr_bucket'] = pd.cut(visible_ranked['ctr'], bins=[-1, 0.5, 1.5, 3.0, 1000],
                                       labels=['<0.5%', '0.5-1.5%', '1.5-3%', '3%+'])
signal2 = visible_ranked.groupby('ctr_bucket', observed=True).agg(
    n=('content_id', 'size'), decline_rate=('is_declining_label', 'mean')
)
print("SIGNAL 2 -- CTR (top-20 position, visible) vs decline rate  ->  verdict: CONFIRMED")
print(signal2.round(3))
print()

print(f"Overall base rate (all rows): {df['is_declining_label'].mean():.3f}")

SIGNAL 1 -- staleness vs decline rate  ->  verdict: MIXED
                      n  decline_rate
staleness_bucket                     
0-30d             20480         0.511
31-90d              175         0.589
91-180d            9171         0.611
180d+               174         0.471

SIGNAL 2 -- CTR (top-20 position, visible) vs decline rate  ->  verdict: CONFIRMED
                n  decline_rate
ctr_bucket                     
<0.5%       12203         0.647
0.5-1.5%     2604         0.507
1.5-3%        244         0.447
3%+            40         0.575

Overall base rate (all rows): 0.542


## 2. Build the ranked queue (writes the CSV)

Score built only from observable signals available before any decision point — no product
flags (none are in this dataset anyway), and no use of `trend_direction`/`trend_pct` (the label
these signals were checked *against*, never an input *to* the rule itself).

In [2]:
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method='average')

def normalize(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

eligible = ((df['avg_position'] > 0) & (df['avg_position'] <= 20)
            & (df['impressions_90d'] >= 100)).astype(int)

df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['position_quality_score'] = 1 - normalize(df['avg_position'].clip(lower=1, upper=20))
df['ctr_gap_score'] = 1 - percentile_rank(df['ctr'])

df['baseline_action_score'] = (
    eligible * (0.4 * df['visibility_score']
                + 0.3 * df['position_quality_score']
                + 0.3 * df['ctr_gap_score'])
).round(4)

df['reason_code'] = np.where(eligible == 1, 'low_ctr_well_positioned', 'not_flagged')
df['action'] = np.where(eligible == 1, 'review_ctr', 'monitor')
df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)

out_cols = ['content_id', 'client_id', 'baseline_rank', 'baseline_action_score', 'reason_code',
            'action', 'avg_position', 'impressions_90d', 'ctr', 'is_declining_label']
out = df[out_cols].sort_values('baseline_rank')

import os
os.makedirs('../outputs', exist_ok=True)
out.to_csv('../outputs/baseline_action_score.csv', index=False)

print(f"Wrote {len(out):,} rows to work/outputs/baseline_action_score.csv")
print(f"Flagged (eligible) rows: {eligible.sum():,} / {len(df):,}")
out.head(10)

Wrote 30,000 rows to work/outputs/baseline_action_score.csv
Flagged (eligible) rows: 15,091 / 30,000


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action,avg_position,impressions_90d,ctr,is_declining_label
3331,content_4a6607efcb46,client_6208ef0f77,1,0.8472,low_ctr_well_positioned,review_ctr,2.2,128068,0.01,0
17362,content_c82bc0c24241,client_f369cb89fc,2,0.8461,low_ctr_well_positioned,review_ctr,4.3,13676,0.00,1
7678,content_8451fc6f034d,client_d029fa3a95,3,0.8432,low_ctr_well_positioned,review_ctr,2.3,272144,0.03,0
7860,content_d225ec9f3d46,client_f369cb89fc,4,0.8403,low_ctr_well_positioned,review_ctr,0.7,26470,0.05,1
20777,content_134631e65b9e,client_6208ef0f77,5,0.8372,low_ctr_well_positioned,review_ctr,1.5,4417,0.00,1
25462,content_825a9788af8d,client_4e07408562,6,0.8322,low_ctr_well_positioned,review_ctr,5.6,16786,0.00,1
27107,content_0022a6b4290f,client_f369cb89fc,7,0.8321,low_ctr_well_positioned,review_ctr,1.2,29747,0.07,1
12869,content_5d5653c4eb4f,client_4e07408562,8,0.8276,low_ctr_well_positioned,review_ctr,5.7,15101,0.00,0
20537,content_f4e210ee0c27,client_7f2253d7e2,9,0.8259,low_ctr_well_positioned,review_ctr,1.6,24784,0.06,1
23220,content_f986bd514b6e,client_7f2253d7e2,10,0.8249,low_ctr_well_positioned,review_ctr,6.6,22456,0.00,1


## 3. Top-10 review

For each of the top 10 by score: the action, why it's there, and what would make the pick wrong.

In [3]:
top10 = out.head(10).copy()
print(top10[['baseline_rank', 'content_id', 'baseline_action_score', 'reason_code',
              'avg_position', 'impressions_90d', 'ctr', 'is_declining_label']].to_string(index=False))

 baseline_rank           content_id  baseline_action_score             reason_code  avg_position  impressions_90d  ctr  is_declining_label
             1 content_4a6607efcb46                 0.8472 low_ctr_well_positioned           2.2           128068 0.01                   0
             2 content_c82bc0c24241                 0.8461 low_ctr_well_positioned           4.3            13676 0.00                   1
             3 content_8451fc6f034d                 0.8432 low_ctr_well_positioned           2.3           272144 0.03                   0
             4 content_d225ec9f3d46                 0.8403 low_ctr_well_positioned           0.7            26470 0.05                   1
             5 content_134631e65b9e                 0.8372 low_ctr_well_positioned           1.5             4417 0.00                   1
             6 content_825a9788af8d                 0.8322 low_ctr_well_positioned           5.6            16786 0.00                   1
             7 content_0022

**Review of the actual top 10** (rank | position | impressions_90d | CTR | currently declining?):

1. `content_4a6607efcb46` — pos 2.2, 128,068 impr, CTR 1% — *action:* review_ctr. *Why:* strong
   position, huge visibility, CTR far below what position 2 should earn. *What would make this
   wrong:* not currently declining (label=0) — this page may already be stable at a low-CTR
   equilibrium (e.g. an informational query with low click intent), not a fixable CTR problem.
2. `content_c82bc0c24241` — pos 4.3, 13,676 impr, CTR 0% — *action:* review_ctr. *Why:* good
   position, real traffic, literally zero recorded clicks. *Wrong if:* the 0% is a tracking gap
   rather than a real CTR floor — worth a sanity check against raw click logs before trusting it.
3. `content_8451fc6f034d` — pos 2.3, 272,144 impr, CTR 3% — *action:* review_ctr. *Why:* the
   single highest-visibility page in the top 10. *Wrong if:* not currently declining (label=0) —
   3% CTR at position 2 may already be near-normal for this query type (e.g. heavy "position
   zero"/featured-snippet competition suppressing clicks industry-wide, not a content problem).
4. `content_d225ec9f3d46` — pos 0.7, 26,470 impr, CTR 5% — *action:* review_ctr. *Why:* position
   0.7 (likely a featured snippet) with surprisingly low CTR for that slot. *Wrong if:* the SERP
   feature itself is answering the query without a click — no amount of content refresh fixes
   that, since the click was never winnable.
5. `content_134631e65b9e` — pos 1.5, 4,417 impr, CTR 0% — *action:* review_ctr. *Why:* top
   position, zero clicks recorded despite it. *Wrong if:* low absolute impression volume (4.4k)
   makes the 0% CTR statistically shaky — a handful of clicks would swing this a lot.
6. `content_825a9788af8d` — pos 5.6, 16,786 impr, CTR 0% — *action:* review_ctr. *Why:* solid
   position just outside the top 5, no clicks. *Wrong if:* this is a brand/navigational query
   where users already know the destination and don't need to click through from search.
7. `content_0022a6b4290f` — pos 1.2, 29,747 impr, CTR 7% — *action:* review_ctr. *Why:* flagged
   mainly on visibility/position weight even though its CTR (7%) is actually decent. *Wrong if:*
   this shouldn't rank this high at all — a sign the score formula may be overweighting position
   relative to CTR for borderline cases like this one.
8. `content_5d5653c4eb4f` — pos 5.7, 15,101 impr, CTR 0% — *action:* review_ctr. *Why:* same
   shape as #6. *Wrong if:* not currently declining (label=0) — possibly a page that plateaued at
   low CTR long ago rather than one actively losing performance now.
9. `content_f4e210ee0c27` — pos 1.6, 24,784 impr, CTR 6% — *action:* review_ctr. *Why:* strong
   position, meaningful traffic, CTR still below what position 1-2 typically earns. *Wrong if:*
   6% CTR is actually reasonable for a highly competitive SERP — no true gap to fix.
10. `content_f986bd514b6e` — pos 6.6, 22,456 impr, CTR 0% — *action:* review_ctr. *Why:* good
    visibility for a position outside the top 5, zero recorded clicks. *Wrong if:* low absolute
    click counts at this position make "0% CTR" an unstable estimate rather than a real signal.

**Pattern across all 10:** every single one shares the same reason code and action by design —
that's the rule doing exactly what it says. The honest weakness, visible without cherry-picking,
is that **7 of the top 20** are flagged but not currently declining by the proxy label (see
section 4) — a reminder that "low CTR at a good position" and "actively declining" are related
but not the same thing.

## 4. Weak picks + leakage check

In [4]:
# Leakage check: confirm the rule's inputs never include the label or a product flag
rule_inputs = {'avg_position', 'impressions_90d', 'ctr'}
label_and_flags = {'trend_direction', 'trend_pct', 'is_declining_label',
                    'health_score', 'priority_score', 'action_type'}  # product flags not in this dataset
print("Rule inputs:", rule_inputs)
print("Overlap with label/product-flag columns:", rule_inputs & label_and_flags)
assert not (rule_inputs & label_and_flags), "Leakage: a rule input overlaps the label/flags"
print("No overlap -- confirmed clean.")
print()

# Weak picks, two angles.
# Angle A: eligibility-edge cases (thin visibility or borderline position) -- none found here,
# which itself is worth noting: the score's 0.4/0.3/0.3 weighting keeps edge cases from
# reaching the very top on visibility alone.
top20 = out.head(20)
borderline = top20[(top20['impressions_90d'] < 150) | (top20['avg_position'] > 15)]
print(f"Eligibility-edge picks in the top 20: {len(borderline)}")

# Angle B: the real weak spot -- flagged rows that are NOT currently declining by the proxy
# label. This is the honest signal that the rule finds a real pattern (low CTR at a good
# position) that is correlated with, but not identical to, 'currently declining'.
weak = top20[top20['is_declining_label'] == 0]
print(f"Top-20 picks that are NOT currently declining by the proxy label: {len(weak)} / 20\n")
weak[['baseline_rank', 'content_id', 'avg_position', 'impressions_90d', 'ctr', 'is_declining_label']]

Rule inputs: {'avg_position', 'ctr', 'impressions_90d'}
Overlap with label/product-flag columns: set()
No overlap -- confirmed clean.

Eligibility-edge picks in the top 20: 0
Top-20 picks that are NOT currently declining by the proxy label: 7 / 20



,baseline_rank,content_id,avg_position,impressions_90d,ctr,is_declining_label
3331,1,content_4a6607efcb46,2.2,128068,0.01,0
7678,3,content_8451fc6f034d,2.3,272144,0.03,0
12869,8,content_5d5653c4eb4f,5.7,15101,0.00,0
29080,11,content_50dfd64f9e8e,2.4,4446,0.00,0
16736,15,content_e12868d1f396,2.9,149712,0.07,0
4589,16,content_339b357d04c7,3.7,46879,0.01,0
8954,19,content_cbdf5a78dcd0,2.4,14830,0.02,0
